# 📊 Text Simplification: Comprehensive Evaluation & Analysis

**Project:** Meaning Preservation and Text Simplification  
**Model:** `google/flan-t5-base`  
**Dataset:** Kaggle Complex-Simple Pairs (`comp-simp1.csv`)  
**Metrics:** SARI, BLEU, Flesch Reading Ease, Flesch-Kincaid Grade Level, Semantic Similarity  

---
This notebook performs a **full statistical analysis and evaluation** of the text simplification pipeline, including:
1. Exploratory Data Analysis (EDA) with 4+ visualisations
2. Model inference on 50 samples
3. Multi-metric evaluation (SARI, BLEU, Readability, Meaning Preservation)
4. Result export to CSV

## 0. Install & Import Dependencies

In [ ]:
!pip install -q pandas matplotlib seaborn transformers torch sacrebleu textstat evaluate sentence-transformers

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import textstat
import evaluate
import warnings
warnings.filterwarnings('ignore')

from transformers import T5ForConditionalGeneration, T5Tokenizer
from sacrebleu.metrics import BLEU
from sentence_transformers import SentenceTransformer, util

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print('✅ All libraries loaded successfully!')

## 1. Data Loading & Preprocessing
We use the **Kaggle Complex-Simple Pairs dataset** (`comp-simp1.csv`) which contains paired complex and simplified sentences.

In [ ]:
FILE_PATH = '../backend/data/comp-simp1.csv'

df = pd.read_csv(FILE_PATH)

# The dataset columns are 'input' (complex) and 'output' (simplified reference)
df = df[['input', 'output']].dropna().reset_index(drop=True)
df.columns = ['complex_text', 'reference_simple']

# Add word counts
df['complex_word_count']    = df['complex_text'].apply(lambda x: len(str(x).split()))
df['reference_word_count']  = df['reference_simple'].apply(lambda x: len(str(x).split()))
df['compression_ratio']     = df['reference_word_count'] / df['complex_word_count'].replace(0, 1)

# Add readability scores
df['complex_fre']           = df['complex_text'].apply(textstat.flesch_reading_ease)
df['reference_fre']         = df['reference_simple'].apply(textstat.flesch_reading_ease)
df['complex_grade']         = df['complex_text'].apply(textstat.flesch_kincaid_grade)
df['reference_grade']        = df['reference_simple'].apply(textstat.flesch_kincaid_grade)

print(f'✅ Dataset loaded: {len(df):,} samples')
print(f'Average complex word count:   {df["complex_word_count"].mean():.1f}')
print(f'Average reference word count: {df["reference_word_count"].mean():.1f}')
df.head()

## 2. Exploratory Data Analysis (EDA)
### 2.1 Word Count Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Word Count Distribution: Complex vs. Simple', fontsize=15, fontweight='bold')

axes[0].hist(df['complex_word_count'].clip(0, 60), bins=30, color='#6366f1', alpha=0.8, edgecolor='white')
axes[0].set_title('Complex Sentences')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['reference_word_count'].clip(0, 60), bins=30, color='#22c55e', alpha=0.8, edgecolor='white')
axes[1].set_title('Reference Simplified Sentences')
axes[1].set_xlabel('Word Count')

plt.tight_layout()
plt.savefig('plot1_word_count_distribution.png', dpi=150)
plt.show()
print(f'Complex mean: {df["complex_word_count"].mean():.1f} | Simple mean: {df["reference_word_count"].mean():.1f}')

### 2.2 Readability Score Comparison (Flesch Reading Ease)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sample = df.sample(min(100, len(df)), random_state=42).reset_index(drop=True)
x = range(len(sample))

ax.fill_between(x, sample['complex_fre'].clip(-20, 120), alpha=0.4, color='#6366f1', label='Complex FRE')
ax.fill_between(x, sample['reference_fre'].clip(-20, 120), alpha=0.4, color='#22c55e', label='Reference Simple FRE')
ax.axhline(y=60, color='gray', linestyle='--', linewidth=1, label='Readability threshold (60)')

ax.set_title('Flesch Reading Ease: Complex vs. Simple Sentences', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Flesch Reading Ease (higher = easier)')
ax.legend()

plt.tight_layout()
plt.savefig('plot2_readability_comparison.png', dpi=150)
plt.show()
print(f'Mean Complex FRE: {df["complex_fre"].mean():.1f} | Mean Reference FRE: {df["reference_fre"].mean():.1f}')

### 2.3 Compression Ratio Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ratios = df['compression_ratio'].clip(0, 2)
ax.hist(ratios, bins=40, color='#a855f7', alpha=0.85, edgecolor='white')
ax.axvline(x=ratios.mean(), color='#ef4444', linestyle='--', linewidth=2, label=f'Mean: {ratios.mean():.2f}')
ax.axvline(x=1.0, color='#64748b', linestyle=':', linewidth=1.5, label='No change (ratio=1)')

ax.set_title('Compression Ratio Distribution (Simple / Complex Words)', fontsize=14, fontweight='bold')
ax.set_xlabel('Ratio (< 1 means shorter output)')
ax.set_ylabel('Frequency')
ax.legend()

plt.tight_layout()
plt.savefig('plot3_compression_ratio.png', dpi=150)
plt.show()
print(f'{(ratios < 1).mean()*100:.1f}% of reference samples are shorter than original')

### 2.4 Grade Level (Flesch-Kincaid) Before vs After

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bins = np.linspace(-5, 25, 30)
ax.hist(df['complex_grade'].clip(-5, 25), bins=bins, alpha=0.7, color='#6366f1', label='Complex', edgecolor='white')
ax.hist(df['reference_grade'].clip(-5, 25), bins=bins, alpha=0.7, color='#22c55e', label='Reference Simple', edgecolor='white')

ax.set_title('Flesch-Kincaid Grade Level Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Grade Level (lower = easier)')
ax.set_ylabel('Frequency')
ax.legend()

plt.tight_layout()
plt.savefig('plot4_grade_level.png', dpi=150)
plt.show()
print(f'Complex Grade: {df["complex_grade"].mean():.1f} | Reference Grade: {df["reference_grade"].mean():.1f}')

## 3. Model Loading
Using the upgraded **FLAN-T5 Base** model — the same model powering the live web application.

In [ ]:
MODEL_NAME = 'google/flan-t5-base'
print(f'Loading {MODEL_NAME}...')

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'✅ Model loaded on {device}')

def simplify_text(text):
    prompt = f'Rewrite the following sentence in very simple words so a child can understand: {str(text).strip()}'
    inputs = tokenizer.encode(prompt, return_tensors='pt', max_length=512, truncation=True).to(device)
    outputs = model.generate(
        inputs,
        max_length=150,
        min_length=10,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        repetition_penalty=2.5,
        no_repeat_ngram_size=2
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print('✅ Inference function ready')

## 4. Run Inference on 50 Samples from Kaggle Dataset

In [ ]:
eval_df = df.sample(50, random_state=42).copy().reset_index(drop=True)

print('Running simplification on 50 samples...')
results = []
for i, row in eval_df.iterrows():
    simplified = simplify_text(row['complex_text'])
    results.append(simplified)
    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/50...')

eval_df['model_output'] = results
eval_df['model_word_count'] = eval_df['model_output'].apply(lambda x: len(str(x).split()))
eval_df['model_fre'] = eval_df['model_output'].apply(textstat.flesch_reading_ease)
eval_df['model_grade'] = eval_df['model_output'].apply(textstat.flesch_kincaid_grade)

print('✅ Inference complete!')
eval_df[['complex_text', 'reference_simple', 'model_output']].head(5)

## 5. Metric Evaluation
### 5.1 SARI, BLEU, and Semantic Similarity

In [ ]:
print('Calculating evaluation metrics...')

# --- BLEU ---
bleu_metric = BLEU(effective_order=True)
hypotheses = eval_df['model_output'].tolist()
references = [eval_df['reference_simple'].tolist()]
bleu_score = bleu_metric.corpus_score(hypotheses, references)

# --- SARI ---
sari_metric = evaluate.load('sari')
sari_result = sari_metric.compute(
    sources=eval_df['complex_text'].tolist(),
    predictions=eval_df['model_output'].tolist(),
    references=[[r] for r in eval_df['reference_simple'].tolist()]
)

# --- Semantic Similarity ---
print('Computing semantic similarity (meaning preservation)...')
sim_model = SentenceTransformer('paraphrase-MiniLM-L3-v2')
emb_complex = sim_model.encode(eval_df['complex_text'].tolist(), convert_to_tensor=True)
emb_output  = sim_model.encode(eval_df['model_output'].tolist(),  convert_to_tensor=True)
similarities = util.pytorch_cos_sim(emb_complex, emb_output).diagonal()
eval_df['semantic_similarity'] = similarities.cpu().numpy()

avg_sim   = eval_df['semantic_similarity'].mean() * 100
avg_fre   = eval_df['model_fre'].mean()
avg_grade = eval_df['model_grade'].mean()

print()
print('=' * 40)
print('       EVALUATION RESULTS (n=50)')
print('=' * 40)
print(f'  BLEU Score:           {bleu_score.score:.2f}')
print(f'  SARI Score:           {sari_result["sari"]:.2f}')
print(f'  Meaning Preservation: {avg_sim:.1f}%')
print(f'  Flesch Reading Ease:  {avg_fre:.1f}')
print(f'  FK Grade Level:       {avg_grade:.1f}')
print('=' * 40)

### 5.2 Results Visualisation

In [ ]:
# Bar chart of aggregate metrics
metrics = {'BLEU': bleu_score.score, 'SARI': sari_result['sari'],
           'Meaning\nPreservation (%)': avg_sim, 'Flesch\nReading Ease': avg_fre}
colors  = ['#6366f1', '#22c55e', '#f59e0b', '#3b82f6']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white', linewidth=0.8, width=0.5)

for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Aggregate Evaluation Metrics (50 Kaggle Samples)', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, max(metrics.values()) * 1.2)
plt.tight_layout()
plt.savefig('plot5_metric_summary.png', dpi=150)
plt.show()

In [ ]:
# Readability before vs after model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Readability: Input vs Model Output', fontsize=14, fontweight='bold')

# FRE
axes[0].hist(eval_df['complex_fre'].clip(-20, 120), bins=20, color='#6366f1', alpha=0.7, label='Complex')
axes[0].hist(eval_df['model_fre'].clip(-20, 120),   bins=20, color='#22c55e', alpha=0.7, label='Model Output')
axes[0].set_title('Flesch Reading Ease')
axes[0].set_xlabel('Score (higher = easier)')
axes[0].legend()

# Grade level
axes[1].hist(eval_df['complex_grade'].clip(-5, 20), bins=20, color='#6366f1', alpha=0.7, label='Complex')
axes[1].hist(eval_df['model_grade'].clip(-5, 20),   bins=20, color='#22c55e', alpha=0.7, label='Model Output')
axes[1].set_title('FK Grade Level')
axes[1].set_xlabel('Grade (lower = easier)')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot6_readability_before_after.png', dpi=150)
plt.show()

print(f'FRE improved by: {eval_df["model_fre"].mean() - eval_df["complex_fre"].mean():+.1f}')
print(f'Grade reduced by: {eval_df["complex_grade"].mean() - eval_df["model_grade"].mean():+.1f} levels')

In [ ]:
# Semantic similarity distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(eval_df['semantic_similarity'] * 100, bins=20, color='#a855f7', edgecolor='white', alpha=0.85)
ax.axvline(avg_sim, color='#ef4444', linestyle='--', linewidth=2, label=f'Mean: {avg_sim:.1f}%')
ax.set_title('Semantic Similarity Distribution (Meaning Preservation)', fontsize=14, fontweight='bold')
ax.set_xlabel('Cosine Similarity (%)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('plot7_similarity_distribution.png', dpi=150)
plt.show()

## 6. Export Results

In [ ]:
eval_df.to_csv('evaluation_results.csv', index=False)

summary = pd.DataFrame([{
    'BLEU': round(bleu_score.score, 2),
    'SARI': round(sari_result['sari'], 2),
    'Meaning_Preservation_%': round(avg_sim, 1),
    'Flesch_Reading_Ease': round(avg_fre, 1),
    'FK_Grade_Level': round(avg_grade, 1),
    'Samples': 50
}])
summary.to_csv('evaluation_summary.csv', index=False)

print('✅ Results exported:')
print('   - evaluation_results.csv  (per-sample results)')
print('   - evaluation_summary.csv  (aggregate scores)')
print()
print(summary.T.to_string(header=False))